In [7]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 11.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [ ]:
from pathlib import Path
import yaml

# ============================================================
# VisDrone_Final — Dataset YAML
# ============================================================

DATASET_ROOT = Path(
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final"
)

YAML_PATH = Path("/kaggle/working/visdrone_final.yaml")

# Check dataset
print("Dataset exists:", DATASET_ROOT.exists())
print("Dataset path  :", DATASET_ROOT)

# Check required directories
required_dirs = [
    DATASET_ROOT / "train" / "images",
    DATASET_ROOT / "train" / "labels",
    DATASET_ROOT / "val" / "images",
    DATASET_ROOT / "val" / "labels",
    DATASET_ROOT / "test" / "images",
    DATASET_ROOT / "test" / "labels",
]

for d in required_dirs:
    print(f"{d}: {d.exists()}")

# ============================================================
# YAML configuration
# ============================================================

data = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",

    "nc": 10,

    "names": [
        "pedestrian",
        "people",
        "bicycle",
        "car",
        "van",
        "truck",
        "tricycle",
        "awning-tricycle",
        "bus",
        "motor"
    ]
}

# Write YAML
with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("\n" + "=" * 60)
print("YAML CREATED")
print("=" * 60)

print(YAML_PATH)
print("\nContents:\n")

with open(YAML_PATH, "r") as f:
    print(f.read())

In [ ]:
# ============================================================
# STEP 1 — YOLOv8n BASELINE ON VISDRONE
# Manuscript protocol: AdamW, lr=0.002
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import torch
import pandas as pd
import os
import json
import platform

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------
DATA_YAML = "/kaggle/working/visdrone_final.yaml"

PROJECT_DIR = "/kaggle/working/YOLO_VisDrone_Benchmark"
RUN_NAME = "YOLOv8n_AdamW_seed42"

MODEL_NAME = "yolov8n.pt"

EPOCHS = 100
IMGSZ = 640
BATCH = 8

OPTIMIZER = "AdamW"
LR0 = 0.002
WEIGHT_DECAY = 0.0005

SEED = 42
DEVICE = 0

# ------------------------------------------------------------
# 2. Environment information
# ------------------------------------------------------------
print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(DEVICE)}")
    print(f"CUDA version : {torch.version.cuda}")
    print(f"GPU memory   : {torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3:.2f} GB")

import ultralytics
print(f"Ultralytics  : {ultralytics.__version__}")

print()

# ------------------------------------------------------------
# 3. Check dataset
# ------------------------------------------------------------
assert Path(DATA_YAML).exists(), f"Dataset YAML not found: {DATA_YAML}"

print("=" * 70)
print("DATASET")
print("=" * 70)
print(f"YAML         : {DATA_YAML}")

# ------------------------------------------------------------
# 4. Load YOLOv8n pretrained model
# ------------------------------------------------------------
print()
print("=" * 70)
print("MODEL")
print("=" * 70)

model = YOLO(MODEL_NAME)

print(f"Model        : {MODEL_NAME}")
print("Pretrained   : True")

# ------------------------------------------------------------
# 5. Train
# ------------------------------------------------------------
print()
print("=" * 70)
print("TRAINING")
print("=" * 70)

results = model.train(
    data=DATA_YAML,

    # Training duration
    epochs=EPOCHS,

    # Image / batch
    imgsz=IMGSZ,
    batch=BATCH,

    # Optimizer — manuscript protocol
    optimizer=OPTIMIZER,
    lr0=LR0,
    weight_decay=WEIGHT_DECAY,

    # Reproducibility
    seed=SEED,
    deterministic=True,

    # Hardware
    device=DEVICE,
    workers=2,

    # Validation / checkpointing
    val=True,
    patience=20,
    save=True,
    save_period=-1,

    # Detection
    max_det=1000,

    # AMP
    amp=True,

    # Learning-rate schedule
    cos_lr=True,

    # Warm-up
    warmup_epochs=3,

    # Output
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,

    # Standard Ultralytics augmentation
    # Keep defaults unless the manuscript explicitly specifies otherwise.
)

# ------------------------------------------------------------
# 6. Locate output directory
# ------------------------------------------------------------
RUN_DIR = Path(PROJECT_DIR) / RUN_NAME

print()
print("=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)
print(f"Run directory: {RUN_DIR}")

# ------------------------------------------------------------
# 7. Results CSV
# ------------------------------------------------------------
RESULTS_CSV = RUN_DIR / "results.csv"

if RESULTS_CSV.exists():

    df = pd.read_csv(RESULTS_CSV)

    # Remove whitespace from column names
    df.columns = [c.strip() for c in df.columns]

    print()
    print("=" * 70)
    print("BEST EPOCH")
    print("=" * 70)

    # YOLO results column
    map95_col = "metrics/mAP50-95(B)"

    if map95_col in df.columns:
        best_idx = df[map95_col].idxmax()
        best_row = df.loc[best_idx]

        print(f"Best epoch        : {int(best_row['epoch']) + 1}")
        print(f"Precision         : {best_row['metrics/precision(B)']:.5f}")
        print(f"Recall            : {best_row['metrics/recall(B)']:.5f}")
        print(f"mAP@50            : {best_row['metrics/mAP50(B)']:.5f}")
        print(f"mAP@50-95         : {best_row['metrics/mAP50-95(B)']:.5f}")

    # --------------------------------------------------------
    # 8. Save a compact benchmark record
    # --------------------------------------------------------
    benchmark = {
        "model": "YOLOv8n",
        "dataset": "VisDrone2019",
        "seed": SEED,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "epochs": EPOCHS,
        "optimizer": OPTIMIZER,
        "lr0": LR0,
        "weight_decay": WEIGHT_DECAY,
        "deterministic": True,
        "pretrained": True,
        "results_csv": str(RESULTS_CSV),
    }

    if map95_col in df.columns:
        benchmark.update({
            "best_epoch": int(best_row["epoch"]) + 1,
            "precision": float(best_row["metrics/precision(B)"]),
            "recall": float(best_row["metrics/recall(B)"]),
            "mAP50": float(best_row["metrics/mAP50(B)"]),
            "mAP50_95": float(best_row["metrics/mAP50-95(B)"]),
        })

    benchmark_path = RUN_DIR / "benchmark_record.json"

    with open(benchmark_path, "w") as f:
        json.dump(benchmark, f, indent=4)

    print()
    print("=" * 70)
    print("FILES")
    print("=" * 70)
    print(f"Results CSV     : {RESULTS_CSV}")
    print(f"Benchmark JSON  : {benchmark_path}")

else:
    print("WARNING: results.csv was not found.")

# ------------------------------------------------------------
# 9. Final validation using BEST checkpoint
# ------------------------------------------------------------
BEST_PT = RUN_DIR / "weights" / "best.pt"

if BEST_PT.exists():

    print()
    print("=" * 70)
    print("FINAL VALIDATION — BEST CHECKPOINT")
    print("=" * 70)

    best_model = YOLO(str(BEST_PT))

    val_results = best_model.val(
        data=DATA_YAML,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        split="val",
        max_det=1000,
        plots=True,
        save_json=False,
        verbose=True
    )

    print()
    print("=" * 70)
    print("FINAL BENCHMARK")
    print("=" * 70)

    print(f"Precision    : {val_results.box.mp:.5f}")
    print(f"Recall       : {val_results.box.mr:.5f}")
    print(f"mAP@50       : {val_results.box.map50:.5f}")
    print(f"mAP@50-95    : {val_results.box.map:.5f}")

    print()
    print("Per-class results:")
    print(val_results.results_dict)

else:
    print(f"WARNING: best.pt not found at {BEST_PT}")

print()
print("=" * 70)
print("STEP 1 COMPLETE")
print("=" * 70)

In [1]:
# ================================================================
# YOLOv8n + SKA
# CONTROLLED ARCHITECTURE-ONLY EXPERIMENT
# ================================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO


# ================================================================
# 1. CONFIGURATION
# ================================================================

DATA_YAML = "/kaggle/working/visdrone_final.yaml"

EPOCHS = 100
IMGSZ = 640
BATCH = 8
DEVICE = 0
SEED = 42

PROJECT = "/kaggle/working/YOLO_VisDrone_Benchmark"
RUN_NAME = "YOLOv8n_SKA_AdamW_seed42"


# ================================================================
# 2. REPRODUCIBILITY
# ================================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ================================================================
# 3. SELECTIVE KERNEL ATTENTION (SKA)
# ================================================================
#
# Input
#   |
#   +---- 3x3 convolution ----+
#   |                         |
#   +---- 5x5 convolution ----+----> U
#                             |
#                       Global Average Pool
#                             |
#                       Channel reduction
#                             |
#                       Branch attention
#                             |
#                    Softmax over branches
#                             |
#                 Weighted branch fusion
#                             |
#                           Output
#
# Two branches provide different receptive fields.
# The attention mechanism learns which branch to emphasize.
# ================================================================

class SKA(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16
    ):
        super().__init__()

        # ----------------------------------------------------------
        # Branch 1: 3x3
        # ----------------------------------------------------------

        self.branch3 = nn.Sequential(
            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                stride=1,
                padding=1,
                groups=1,
                bias=False
            ),
            nn.BatchNorm2d(channels),
            nn.SiLU(inplace=True)
        )

        # ----------------------------------------------------------
        # Branch 2: 5x5
        # ----------------------------------------------------------

        self.branch5 = nn.Sequential(
            nn.Conv2d(
                channels,
                channels,
                kernel_size=5,
                stride=1,
                padding=2,
                groups=1,
                bias=False
            ),
            nn.BatchNorm2d(channels),
            nn.SiLU(inplace=True)
        )

        # ----------------------------------------------------------
        # Attention generation
        # ----------------------------------------------------------

        hidden = max(channels // reduction, 1)

        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(
                channels,
                hidden,
                kernel_size=1,
                bias=False
            ),
            nn.SiLU(inplace=True)
        )

        # Two branch attention vectors
        self.attention = nn.Conv2d(
            hidden,
            channels * 2,
            kernel_size=1,
            bias=True
        )

        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):

        # ----------------------------------------------------------
        # Generate two receptive-field representations
        # ----------------------------------------------------------

        x3 = self.branch3(x)
        x5 = self.branch5(x)

        # ----------------------------------------------------------
        # Fuse representations for attention generation
        # ----------------------------------------------------------

        u = x3 + x5

        # ----------------------------------------------------------
        # Global descriptor
        # ----------------------------------------------------------

        s = self.global_pool(u)

        z = self.fc(s)

        # ----------------------------------------------------------
        # Generate branch attention
        #
        # Shape before reshape:
        #   B x (2C) x 1 x 1
        #
        # After reshape:
        #   B x 2 x C x 1 x 1
        # ----------------------------------------------------------

        a = self.attention(z)

        b, _, _, _ = a.shape

        a = a.view(
            b,
            2,
            -1,
            1,
            1
        )

        # Softmax across the two branches
        a = self.softmax(a)

        # ----------------------------------------------------------
        # Selective fusion
        # ----------------------------------------------------------

        x3 = x3.unsqueeze(1)
        x5 = x5.unsqueeze(1)

        branches = torch.cat(
            [x3, x5],
            dim=1
        )

        out = (
            branches * a
        ).sum(dim=1)

        # Residual connection
        out = out + x

        return out


# ================================================================
# 4. LOAD FRESH PRETRAINED YOLOv8n
# ================================================================

print("=" * 70)
print("LOADING FRESH PRETRAINED YOLOv8n")
print("=" * 70)

model = YOLO("yolov8n.pt")

print("YOLOv8n pretrained model loaded.")


# ================================================================
# 5. INSERT SKA
# ================================================================
#
# YOLOv8n actual channels:
#
# Layer 4 -> P3 -> 64
# Layer 6 -> P4 -> 128
# Layer 8 -> P5 -> 256
#
# SKA is inserted AFTER each C2f block.
# ================================================================

target_layers = {
    4: 64,
    6: 128,
    8: 256
}

ultralytics_model = model.model

for idx, channels in target_layers.items():

    original_layer = ultralytics_model.model[idx]

    ska = SKA(
        channels=channels,
        reduction=16
    )

    wrapped_layer = nn.Sequential(
        original_layer,
        ska
    )

    # Preserve required Ultralytics metadata
    wrapped_layer.i = original_layer.i
    wrapped_layer.f = original_layer.f
    wrapped_layer.type = original_layer.type

    # DO NOT use original_layer.np
    # It is not present in this Ultralytics version.

    ultralytics_model.model[idx] = wrapped_layer


# ================================================================
# 6. VERIFY SKA INSERTION
# ================================================================

print("\n" + "=" * 70)
print("SKA ARCHITECTURE")
print("=" * 70)

print("Layer 4 : C2f + SKA(64)")
print("Layer 6 : C2f + SKA(128)")
print("Layer 8 : C2f + SKA(256)")

ska_count = sum(
    1
    for module in ultralytics_model.model.modules()
    if isinstance(module, SKA)
)

print(f"\nSKA modules detected: {ska_count}")


# ================================================================
# 7. PARAMETER COUNT
# ================================================================

base_model = YOLO("yolov8n.pt")

base_params = sum(
    p.numel()
    for p in base_model.model.parameters()
)

ska_params = sum(
    p.numel()
    for p in ultralytics_model.parameters()
)

additional_params = ska_params - base_params

increase_percent = (
    additional_params / base_params
) * 100

print("\n" + "=" * 70)
print("PARAMETER COMPARISON")
print("=" * 70)

print(f"YOLOv8n              : {base_params:,}")
print(f"YOLOv8n + SKA        : {ska_params:,}")
print(f"Additional parameters: {additional_params:,}")
print(f"Increase             : {increase_percent:.3f}%")


# ================================================================
# 8. DUMMY FORWARD TEST
# ================================================================

print("\n" + "=" * 70)
print("DUMMY FORWARD TEST")
print("=" * 70)

ultralytics_model.to(DEVICE)

dummy = torch.randn(
    1,
    3,
    IMGSZ,
    IMGSZ,
    device=DEVICE
)

with torch.no_grad():

    _ = ultralytics_model(dummy)

print("Input shape : (1, 3, 640, 640)")
print("Forward pass: SUCCESS")


# ================================================================
# 9. MODEL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("MODEL SUMMARY")
print("=" * 70)

model.info(verbose=True)


# ================================================================
# 10. TRAINING CONFIGURATION
# ================================================================

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(f"Dataset             : {DATA_YAML}")
print(f"Model               : YOLOv8n + SKA")
print(f"Epochs              : {EPOCHS}")
print(f"Image size          : {IMGSZ}")
print(f"Batch size          : {BATCH}")
print(f"Optimizer           : AdamW")
print(f"Learning rate       : 0.002")
print(f"Weight decay        : 0.0005")
print(f"Cosine LR           : True")
print(f"Warmup epochs       : 3")
print(f"AMP                 : True")
print(f"Seed                : {SEED}")
print(f"Deterministic       : True")
print(f"Max detections      : 1000")
print(f"Copy-Paste          : OFF")
print(f"Hyperparameter tune : OFF")
print(f"Device              : {DEVICE}")


# ================================================================
# 11. TRAIN
# ================================================================

print("\n" + "=" * 70)
print("STARTING YOLOv8n + SKA TRAINING")
print("=" * 70)

results = model.train(

    data=DATA_YAML,

    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,

    device=DEVICE,

    optimizer="AdamW",

    lr0=0.002,
    weight_decay=0.0005,

    cos_lr=True,
    warmup_epochs=3,

    amp=True,

    seed=SEED,
    deterministic=True,

    patience=20,

    max_det=1000,

    workers=2,

    pretrained=True,

    # Architecture-only comparison
    copy_paste=0.0,

    project=PROJECT,
    name=RUN_NAME,

    exist_ok=True,

    verbose=True
)

print("\n" + "=" * 70)
print("SKA TRAINING COMPLETED")
print("=" * 70)

RUN_DIR = os.path.join(
    PROJECT,
    RUN_NAME
)

print(f"Run directory : {RUN_DIR}")
print(f"Best model    : {RUN_DIR}/weights/best.pt")
print(f"Results CSV   : {RUN_DIR}/results.csv")

ModuleNotFoundError: No module named 'ultralytics'

In [2]:
from ultralytics import YOLO
runs = {  # name: path to best.pt
    "YOLOv8n": "/kaggle/working/YOLO_VisDrone_Benchmark/YOLOv8n_AdamW_seed42/weights/best.pt",
    # "YOLOv8n_SKA": "/kaggle/working/YOLO_VisDrone_Benchmark/YOLOv8n_SKA_AdamW_seed42/weights/best.pt
    # add the other models here
}
for name, w in runs.items():
    YOLO(w).val(data="visdrone_final.yaml", split="test", imgsz=640,
                conf=0.001, iou=0.7, max_det=500, save_json=True, name=name)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.18 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,134,894 parameters, 0 gradients, 8.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.1 ms, read: 3.7±2.1 MB/s, size: 258.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 19.7it/s 1:220.1sss
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/

In [3]:
# ============================================================
# SIZE-STRATIFIED AP / RECALL EVALUATION
# ============================================================

!pip install -q pycocotools pillow

import argparse
import csv
import json
from pathlib import Path

import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


BINS = {
    "coco": [
        ("small", 0, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],

    "fine": [
        ("tiny", 0, 16**2),
        ("small", 16**2, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],
}


def build_gt(img_dir, lbl_dir, n_classes, cat_offset):

    images = []
    anns = []
    aid = 1

    for p in sorted(Path(img_dir).iterdir()):

        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        w, h = Image.open(p).size

        iid = int(p.stem) if p.stem.isnumeric() else p.stem

        images.append(
            dict(
                id=iid,
                width=w,
                height=h,
                file_name=p.name
            )
        )

        lf = Path(lbl_dir) / (p.stem + ".txt")

        if not lf.exists():
            continue

        for line in lf.read_text().strip().splitlines():

            parts = line.split()

            if len(parts) < 5:
                continue

            c = int(float(parts[0]))

            xc, yc, bw, bh = (
                float(v) for v in parts[1:5]
            )

            # Convert normalized YOLO coordinates
            # to ORIGINAL image pixel coordinates
            bw = bw * w
            bh = bh * h

            x = xc * w - bw / 2
            y = yc * h - bh / 2

            anns.append(
                dict(
                    id=aid,
                    image_id=iid,
                    category_id=c + cat_offset,
                    bbox=[x, y, bw, bh],
                    area=bw * bh,
                    iscrowd=0
                )
            )

            aid += 1

    cats = [
        dict(
            id=i + cat_offset,
            name=str(i)
        )
        for i in range(n_classes)
    ]

    gt = COCO()

    gt.dataset = dict(
        images=images,
        annotations=anns,
        categories=cats
    )

    gt.createIndex()

    return gt


def mean_valid(a):

    a = a[a > -1]

    return float(a.mean()) if a.size else float("nan")


def evaluate(gt, pred_file, bins, maxdet):

    with open(pred_file, "r") as f:
        preds = json.load(f)

    dt = gt.loadRes(preds)

    E = COCOeval(
        gt,
        dt,
        "bbox"
    )

    E.params.maxDets = [10, 100, maxdet]

    E.params.areaRng = (
        [[0, 1e10]]
        + [[lo, hi] for _, lo, hi in bins]
    )

    E.params.areaRngLbl = (
        ["all"]
        + [b[0] for b in bins]
    )

    E.evaluate()
    E.accumulate()

    prec = E.eval["precision"]
    rec = E.eval["recall"]

    # maxDet index
    m = 2

    rows = []

    areas = np.array([
        a["area"]
        for a in gt.dataset["annotations"]
    ])

    for a, lbl in enumerate(E.params.areaRngLbl):

        lo, hi = E.params.areaRng[a]

        n = int(
            ((areas >= lo) & (areas <= hi)).sum()
        )

        rows.append(
            dict(
                bin=lbl,
                n_gt=n,

                AP50_95=mean_valid(
                    prec[:, :, :, a, m]
                ),

                AP50=mean_valid(
                    prec[0, :, :, a, m]
                ),

                AR50_95=mean_valid(
                    rec[:, :, a, m]
                ),

                AR50=mean_valid(
                    rec[0, :, a, m]
                )
            )
        )

    return rows


def main(argv=None):

    ap = argparse.ArgumentParser()

    ap.add_argument(
        "--images",
        required=True
    )

    ap.add_argument(
        "--labels",
        required=True
    )

    ap.add_argument(
        "--runs",
        nargs="+",
        required=True,
        help="NAME=path/to/predictions.json"
    )

    ap.add_argument(
        "--nc",
        type=int,
        default=10
    )

    ap.add_argument(
        "--bins",
        choices=BINS,
        default="fine"
    )

    ap.add_argument(
        "--maxdet",
        type=int,
        default=500
    )

    ap.add_argument(
        "--cat_offset",
        type=int,
        default=-1,
        help="0 or 1; -1 = infer from predictions"
    )

    ap.add_argument(
        "--out",
        default="size_results.csv"
    )

    a = ap.parse_args(argv)

    runs = dict(
        r.split("=", 1)
        for r in a.runs
    )

    # --------------------------------------------------------
    # Infer category offset
    # --------------------------------------------------------

    off = a.cat_offset

    if off < 0:

        first_file = next(
            iter(runs.values())
        )

        with open(first_file, "r") as f:
            first = json.load(f)

        off = min(
            p["category_id"]
            for p in first
        )

        print(
            "Inferred category offset:",
            off
        )

    # --------------------------------------------------------
    # Build ground truth
    # --------------------------------------------------------

    gt = build_gt(
        a.images,
        a.labels,
        a.nc,
        off
    )

    print(
        f"GT: {len(gt.dataset['images'])} images, "
        f"{len(gt.dataset['annotations'])} boxes"
    )

    # --------------------------------------------------------
    # Evaluate models
    # --------------------------------------------------------

    out = []

    for name, f in runs.items():

        print()
        print("=" * 70)
        print(f"Evaluating: {name}")
        print(f"Prediction file: {f}")
        print("=" * 70)

        for r in evaluate(
            gt,
            f,
            BINS[a.bins],
            a.maxdet
        ):

            r["model"] = name

            out.append(r)

            print(
                f"{name:12s} "
                f"{r['bin']:7s} "
                f"n={r['n_gt']:6d} "
                f"AP50={r['AP50']:.3f} "
                f"AP50-95={r['AP50_95']:.3f} "
                f"AR50={r['AR50']:.3f} "
                f"AR50-95={r['AR50_95']:.3f}"
            )

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    with open(
        a.out,
        "w",
        newline=""
    ) as fh:

        w = csv.DictWriter(
            fh,
            fieldnames=[
                "model",
                "bin",
                "n_gt",
                "AP50",
                "AP50_95",
                "AR50",
                "AR50_95"
            ]
        )

        w.writeheader()
        w.writerows(out)

    print()
    print("=" * 70)
    print("Saved:")
    print(a.out)
    print("=" * 70)


# ============================================================
# KAGGLE CONFIGURATION
# ============================================================

main([
    "--images",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/images",

    "--labels",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels",

    "--runs",

    "YOLOv8n=/kaggle/working/runs/detect/YOLOv8n/predictions.json",

    # "YOLOv9t=/kaggle/working/runs/detect/v9t/predictions.json",

    "--nc",
    "10",

    "--bins",
    "fine",

    "--maxdet",
    "500",

    "--cat_offset",
    "-1",

    "--out",
    "/kaggle/working/size_results.csv"
])

Inferred category offset: 1
creating index...
index created!
GT: 1610 images, 75102 boxes

Evaluating: YOLOv8n
Prediction file: /kaggle/working/runs/detect/YOLOv8n/predictions.json
Loading and preparing results...
DONE (t=5.94s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=243.67s).
Accumulating evaluation results...
DONE (t=21.70s).
YOLOv8n      all     n= 75102 AP50=0.270 AP50-95=0.151 AR50=0.576 AR50-95=0.315
YOLOv8n      tiny    n= 27346 AP50=0.083 AP50-95=0.029 AR50=0.276 AR50-95=0.102
YOLOv8n      small   n= 23491 AP50=0.217 AP50-95=0.093 AR50=0.589 AR50-95=0.275
YOLOv8n      medium  n= 21856 AP50=0.410 AP50-95=0.232 AR50=0.778 AR50-95=0.454
YOLOv8n      large   n=  2409 AP50=0.537 AP50-95=0.361 AR50=0.882 AR50-95=0.606

Saved:
/kaggle/working/size_results.csv


In [8]:
from ultralytics import YOLO
runs = {  # name: path to best.pt
    "YOLOv8n_SKA": "/kaggle/working/YOLO_VisDrone_Benchmark/YOLOv8n_SKA_AdamW_seed42/weights/best.pt"
    # add the other models here
}
for name, w in runs.items():
    YOLO(w).val(data="visdrone_final.yaml", split="test", imgsz=640,
                conf=0.001, iou=0.7, max_det=500, save_json=True, name=name)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,134,894 parameters, 0 gradients, 8.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.1 ms, read: 2.4±1.1 MB/s, size: 157.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 21.4it/s 1:150.1sss
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/t

In [9]:
# ============================================================
# SIZE-STRATIFIED AP / RECALL EVALUATION
# ============================================================

!pip install -q pycocotools pillow

import argparse
import csv
import json
from pathlib import Path

import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


BINS = {
    "coco": [
        ("small", 0, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],

    "fine": [
        ("tiny", 0, 16**2),
        ("small", 16**2, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],
}


def build_gt(img_dir, lbl_dir, n_classes, cat_offset):

    images = []
    anns = []
    aid = 1

    for p in sorted(Path(img_dir).iterdir()):

        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        w, h = Image.open(p).size

        iid = int(p.stem) if p.stem.isnumeric() else p.stem

        images.append(
            dict(
                id=iid,
                width=w,
                height=h,
                file_name=p.name
            )
        )

        lf = Path(lbl_dir) / (p.stem + ".txt")

        if not lf.exists():
            continue

        for line in lf.read_text().strip().splitlines():

            parts = line.split()

            if len(parts) < 5:
                continue

            c = int(float(parts[0]))

            xc, yc, bw, bh = (
                float(v) for v in parts[1:5]
            )

            # Convert normalized YOLO coordinates
            # to ORIGINAL image pixel coordinates
            bw = bw * w
            bh = bh * h

            x = xc * w - bw / 2
            y = yc * h - bh / 2

            anns.append(
                dict(
                    id=aid,
                    image_id=iid,
                    category_id=c + cat_offset,
                    bbox=[x, y, bw, bh],
                    area=bw * bh,
                    iscrowd=0
                )
            )

            aid += 1

    cats = [
        dict(
            id=i + cat_offset,
            name=str(i)
        )
        for i in range(n_classes)
    ]

    gt = COCO()

    gt.dataset = dict(
        images=images,
        annotations=anns,
        categories=cats
    )

    gt.createIndex()

    return gt


def mean_valid(a):

    a = a[a > -1]

    return float(a.mean()) if a.size else float("nan")


def evaluate(gt, pred_file, bins, maxdet):

    with open(pred_file, "r") as f:
        preds = json.load(f)

    dt = gt.loadRes(preds)

    E = COCOeval(
        gt,
        dt,
        "bbox"
    )

    E.params.maxDets = [10, 100, maxdet]

    E.params.areaRng = (
        [[0, 1e10]]
        + [[lo, hi] for _, lo, hi in bins]
    )

    E.params.areaRngLbl = (
        ["all"]
        + [b[0] for b in bins]
    )

    E.evaluate()
    E.accumulate()

    prec = E.eval["precision"]
    rec = E.eval["recall"]

    # maxDet index
    m = 2

    rows = []

    areas = np.array([
        a["area"]
        for a in gt.dataset["annotations"]
    ])

    for a, lbl in enumerate(E.params.areaRngLbl):

        lo, hi = E.params.areaRng[a]

        n = int(
            ((areas >= lo) & (areas <= hi)).sum()
        )

        rows.append(
            dict(
                bin=lbl,
                n_gt=n,

                AP50_95=mean_valid(
                    prec[:, :, :, a, m]
                ),

                AP50=mean_valid(
                    prec[0, :, :, a, m]
                ),

                AR50_95=mean_valid(
                    rec[:, :, a, m]
                ),

                AR50=mean_valid(
                    rec[0, :, a, m]
                )
            )
        )

    return rows


def main(argv=None):

    ap = argparse.ArgumentParser()

    ap.add_argument(
        "--images",
        required=True
    )

    ap.add_argument(
        "--labels",
        required=True
    )

    ap.add_argument(
        "--runs",
        nargs="+",
        required=True,
        help="NAME=path/to/predictions.json"
    )

    ap.add_argument(
        "--nc",
        type=int,
        default=10
    )

    ap.add_argument(
        "--bins",
        choices=BINS,
        default="fine"
    )

    ap.add_argument(
        "--maxdet",
        type=int,
        default=500
    )

    ap.add_argument(
        "--cat_offset",
        type=int,
        default=-1,
        help="0 or 1; -1 = infer from predictions"
    )

    ap.add_argument(
        "--out",
        default="size_results.csv"
    )

    a = ap.parse_args(argv)

    runs = dict(
        r.split("=", 1)
        for r in a.runs
    )

    # --------------------------------------------------------
    # Infer category offset
    # --------------------------------------------------------

    off = a.cat_offset

    if off < 0:

        first_file = next(
            iter(runs.values())
        )

        with open(first_file, "r") as f:
            first = json.load(f)

        off = min(
            p["category_id"]
            for p in first
        )

        print(
            "Inferred category offset:",
            off
        )

    # --------------------------------------------------------
    # Build ground truth
    # --------------------------------------------------------

    gt = build_gt(
        a.images,
        a.labels,
        a.nc,
        off
    )

    print(
        f"GT: {len(gt.dataset['images'])} images, "
        f"{len(gt.dataset['annotations'])} boxes"
    )

    # --------------------------------------------------------
    # Evaluate models
    # --------------------------------------------------------

    out = []

    for name, f in runs.items():

        print()
        print("=" * 70)
        print(f"Evaluating: {name}")
        print(f"Prediction file: {f}")
        print("=" * 70)

        for r in evaluate(
            gt,
            f,
            BINS[a.bins],
            a.maxdet
        ):

            r["model"] = name

            out.append(r)

            print(
                f"{name:12s} "
                f"{r['bin']:7s} "
                f"n={r['n_gt']:6d} "
                f"AP50={r['AP50']:.3f} "
                f"AP50-95={r['AP50_95']:.3f} "
                f"AR50={r['AR50']:.3f} "
                f"AR50-95={r['AR50_95']:.3f}"
            )

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    with open(
        a.out,
        "w",
        newline=""
    ) as fh:

        w = csv.DictWriter(
            fh,
            fieldnames=[
                "model",
                "bin",
                "n_gt",
                "AP50",
                "AP50_95",
                "AR50",
                "AR50_95"
            ]
        )

        w.writeheader()
        w.writerows(out)

    print()
    print("=" * 70)
    print("Saved:")
    print(a.out)
    print("=" * 70)


# ============================================================
# KAGGLE CONFIGURATION
# ============================================================

main([
    "--images",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/images",

    "--labels",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels",

    "--runs",

    "YOLOv8n_SKA=/kaggle/working/runs/detect/YOLOv8n_SKA/predictions.json",

    # "YOLOv9t=/kaggle/working/runs/detect/v9t/predictions.json",

    "--nc",
    "10",

    "--bins",
    "fine",

    "--maxdet",
    "500",

    "--cat_offset",
    "-1",

    "--out",
    "/kaggle/working/size_results.csv"
])

Inferred category offset: 1
creating index...
index created!
GT: 1610 images, 75102 boxes

Evaluating: YOLOv8n_SKA
Prediction file: /kaggle/working/runs/detect/YOLOv8n_SKA/predictions.json
Loading and preparing results...
DONE (t=3.58s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=201.54s).
Accumulating evaluation results...
DONE (t=13.06s).
YOLOv8n_SKA  all     n= 75102 AP50=0.254 AP50-95=0.142 AR50=0.562 AR50-95=0.307
YOLOv8n_SKA  tiny    n= 27346 AP50=0.072 AP50-95=0.026 AR50=0.266 AR50-95=0.101
YOLOv8n_SKA  small   n= 23491 AP50=0.201 AP50-95=0.085 AR50=0.576 AR50-95=0.264
YOLOv8n_SKA  medium  n= 21856 AP50=0.382 AP50-95=0.217 AR50=0.767 AR50-95=0.444
YOLOv8n_SKA  large   n=  2409 AP50=0.503 AP50-95=0.335 AR50=0.863 AR50-95=0.588

Saved:
/kaggle/working/size_results.csv
